# 02 — Prompts
### LangChain Foundations

In `01_Models/` you called models with a hardcoded `SystemMessage`/`HumanMessage` list. That works
for a one-off script, but breaks the moment you need to reuse the same structure with different
inputs — a different customer name, a different chat history, a different topic. **Prompt templates**
are how you turn a hardcoded conversation into a reusable, parameterized one.

> 📌 **What's new / worth knowing (as of late 2026)**
> - `MessagesPlaceholder` now supports `n_messages=` — it can auto-truncate to the last N messages
>   for you, instead of you writing that slicing logic by hand. Small feature, saves real code.
> - `MessagesPlaceholder(optional=True)` — won't raise a `KeyError` if you forget to pass history;
>   returns an empty list instead. Useful for a chatbot's very first turn.
> - The single biggest thing people get wrong right now: **the convenient tuple syntax
>   `("system", big_prompt)` silently defeats Anthropic's prompt caching.** No error, no warning —
>   your app just quietly pays full price on every call instead of getting a ~90% discount on the
>   repeated part. Section 7 below covers this in detail, because it's the kind of thing that costs
>   real money in production and nobody tells you about it.

## Setup

In [1]:
# %pip install -q langchain langchain-core langchain-openai langchain-groq python-dotenv


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    FewShotChatMessagePromptTemplate,
)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("Environment ready.")


Environment ready.


---
## 1. `PromptTemplate` — the plain-string template

This is the simpler, older primitive: one string with `{placeholders}` in it. You won't reach for
this often once you're doing chat-based work (`ChatPromptTemplate` below is what you'll actually use
90% of the time) — but it's worth seeing once because `ChatPromptTemplate` is built out of these
underneath, and you'll still meet it when formatting a single block of text (e.g. building one
document inside a RAG pipeline, or writing a reusable string to embed).

In [3]:
single_prompt = PromptTemplate(
    input_variables=["domain", "topic"],
    template="You are a helpful {domain} expert. Explain {topic} in simple terms, in 3 sentences.",
)

# .format() -> a plain string
print(single_prompt.format(domain="teacher", topic="neural networks"))

# .invoke() -> a StringPromptValue (LangChain's standard wrapper, works inside chains/pipes)
prompt_value = single_prompt.invoke({"domain": "teacher", "topic": "neural networks"})
print(type(prompt_value), "->", prompt_value.to_string()[:50], "...")


You are a helpful teacher expert. Explain neural networks in simple terms, in 3 sentences.
<class 'langchain_core.prompt_values.StringPromptValue'> -> You are a helpful teacher expert. Explain neural n ...


---
## 2. `ChatPromptTemplate` — the one you'll actually use

Same idea as `PromptTemplate`, but it produces a **list of messages** instead of one string —
matching exactly what chat models expect. Two ways to build one; both produce the same result.

In [4]:
# Shorthand: tuples of (role, template_string) — what you'll write 95% of the time
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful {domain} expert."),
    ("human", "Explain {topic} in simple terms."),
])

prompt_value = chat_template.invoke({"domain": "teacher", "topic": "neural networks"})
message = prompt_value.to_messages()
print(message)
for msg in message:
    print(f"[{msg.__class__.__name__}]: {msg.content}")

[SystemMessage(content='You are a helpful teacher expert.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain neural networks in simple terms.', additional_kwargs={}, response_metadata={})]
[SystemMessage]: You are a helpful teacher expert.
[HumanMessage]: Explain neural networks in simple terms.


In [5]:
# Explicit class-based equivalent — more verbose, but shows what the tuples are shorthand FOR.
# Reach for this when a message needs something beyond a plain template (see Section 7).
explicit_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("You are a helpful {domain} expert."),
    HumanMessagePromptTemplate.from_template("Explain {topic} in simple terms."),
])

assert explicit_template.invoke({"domain": "teacher", "topic": "neural networks"}).to_messages() == \
       chat_template.invoke({"domain": "teacher", "topic": "neural networks"}).to_messages()
print("Identical output — the tuple shorthand IS this, just less typing.")


Identical output — the tuple shorthand IS this, just less typing.


In [6]:
# The real reason this matters: pipe it straight into a model, no manual message-building at all.
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.7)
chain = chat_template | model   # this is LCEL — covered properly in 03_Runnables

response = chain.invoke({"domain": "teacher", "topic": "neural networks"})
print(response.content)


Imagine you are trying to teach a child to recognize a cat.

You wouldn’t give them a complex math formula like, *"If the object has curved ears, whiskers, and weighs between 4–10 pounds, it is a cat."* Instead, you’d show them lots of pictures. You’d show them a cat, say "That's a cat," then show them a dog, say "That's not a cat," and repeat this many times. Eventually, the child’s brain starts to pick up on the *patterns* that make a cat look like a cat.

**A neural network works almost exactly the same way.**

### 1. The Basic Idea: A Web of Simple Decisions
A neural network is made up of tiny, simple units called **neurons** (not real brain cells, but mathematical models inspired by them). These neurons are connected in layers, like a web.

- **Input Layer:** This is where the information comes in. For example, if the network is recognizing a cat, the input might be the pixels of a photo.
- **Hidden Layers:** These are the "thinking" layers. Each neuron in these layers looks at a 

---
## 3. `MessagesPlaceholder` — injecting a variable-length chat history

A template can't hardcode "message 1, message 2, message 3" — conversations grow. `MessagesPlaceholder`
reserves a *slot* that gets filled with however many messages you pass in at call time.

In [7]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful customer support agent."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{query}"),
])

history = [
    ("human", "I ordered a phone case last week"),
    ("ai", "Got it — could you share your order number?"),
]

prompt_value = chat_template.invoke({"chat_history": history, "query": "It still hasn't shipped"})
for m in prompt_value.to_messages():
    print(f"{m.type:9s}: {m.content}")


system   : You are a helpful customer support agent.
human    : I ordered a phone case last week
ai       : Got it — could you share your order number?
human    : It still hasn't shipped


In [8]:
# optional=True — don't crash on a chatbot's very first turn, when there's no history yet
safe_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{query}"),
])

# No chat_history key passed at all — this would raise a KeyError WITHOUT optional=True
first_turn = safe_template.invoke({"query": "Hi, first time here"})
print(first_turn.to_messages())


[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi, first time here', additional_kwargs={}, response_metadata={})]


In [9]:
# n_messages= — auto-truncate to the last N messages, so old turns fall off automatically.
# This is a real cost/context-window control, not just a convenience: every extra message in
# history is tokens you pay for and context the model has to read, on EVERY single call.
capped_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history", n_messages=2),  # keep only the last 2
    ("human", "{query}"),
])

long_history = [
    ("human", "turn 1"), ("ai", "reply 1"),
    ("human", "turn 2"), ("ai", "reply 2"),
    ("human", "turn 3"), ("ai", "reply 3"),
]

result = capped_template.invoke({"chat_history": long_history, "query": "latest question"})
print(result.to_messages())  # only "turn 3" / "reply 3" made it through, plus the new query


[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='turn 3', additional_kwargs={}, response_metadata={}), AIMessage(content='reply 3', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='latest question', additional_kwargs={}, response_metadata={})]


---
## 4. Few-shot prompting — teaching consistent behavior by example

Sometimes describing what you want in the system prompt isn't enough — the model's *tone* or
*format* drifts. Showing 2–3 examples of exactly the input/output pattern you want is often far
more reliable than describing it. This is `FewShotChatMessagePromptTemplate`.

In [10]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

examples = [
    {"input": "My package is late",
     "output": "I'm sorry about the delay! Could you share your order number so I can look into it right away?"},
    {"input": "This product is broken",
     "output": "That's frustrating, I'm sorry. Let's get this sorted — can you tell me what's wrong with it?"},
]

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_template = ChatPromptTemplate.from_messages([
    ("system", "You are a customer support agent. Match the tone of the examples exactly: "
               "warm, apologetic, and always end with one clarifying question."),
    few_shot_prompt,
    ("human", "{input}"),
])

prompt_value = final_template.invoke({"input": "I was charged twice this month"})
for m in prompt_value.to_messages():
    print(f"{m.type:9s}: {m.content}")


system   : You are a customer support agent. Match the tone of the examples exactly: warm, apologetic, and always end with one clarifying question.
human    : My package is late
ai       : I'm sorry about the delay! Could you share your order number so I can look into it right away?
human    : This product is broken
ai       : That's frustrating, I'm sorry. Let's get this sorted — can you tell me what's wrong with it?
human    : I was charged twice this month


---
## 5. Partial prompts — pre-filling values you don't want to pass every time

`.partial()` locks in some variables now, leaving the rest to be filled in later. Genuinely useful
for things like "today's date" or "the persona name" that shouldn't have to be re-typed on every
single call site in your app.

In [11]:
from datetime import date

base_template = PromptTemplate(
    input_variables=["today", "question"],
    template="Today's date is {today}. Answer this question: {question}",
)

# Lock in "today" once...
dated_template = base_template.partial(today=str(date.today()))

# ...every future call only needs to supply "question"
print(dated_template.format(question="How many days until next Monday?"))


Today's date is 2026-09-24. Answer this question: How many days until next Monday?


---
## 6. Composing prompts

You can build a template up from smaller pieces instead of one giant string — handy when a big
system prompt has reusable sections (e.g. a shared "house style" block reused across several
different assistants).

In [12]:
house_style = "Always respond in a warm, concise tone. Never use more than 3 sentences."
role_instruction = "You are a support agent for a telecom company."

# Just string composition — the simplest form of "prompt composition" and often all you need.
system_text = f"{role_instruction}\n\n{house_style}"

composed_template = ChatPromptTemplate.from_messages([
    ("system", system_text),
    ("human", "{query}"),
])

print(composed_template.invoke({"query": "How do I check my data usage?"}).to_messages()[0].content)


You are a support agent for a telecom company.

Always respond in a warm, concise tone. Never use more than 3 sentences.


---
## 7. ⚠️ Production concern: prompt caching (and how the easy syntax defeats it)

This section is *reference material* — it's about Anthropic's Claude models, and you don't have an
Anthropic key. But understand this now, because you'll hit it the moment you do have one, and the
mistake is invisible: no error, no warning, you just silently pay full price forever.

**How prompt caching actually works:** if part of your prompt is identical across many calls (a long
system prompt, a big set of few-shot examples, a large document you keep referencing), the provider
can cache that prefix and charge you a fraction of the price for it on repeat calls — instead of
re-processing it from scratch every single time.

**The catch, provider by provider:**

| Provider | How caching is triggered |
|---|---|
| **OpenAI** | Automatic — any prompt prefix over ~1024 tokens gets cached with zero code changes |
| **Anthropic (Claude)** | **Manual** — you must mark the exact spot with a `cache_control` breakpoint |
| **Groq / Gemini** | Provider-managed, no manual markers needed (behavior varies — check current docs before relying on it) |

The tuple shorthand you've been using all notebook, `("system", big_prompt)`, has **no way to attach
a `cache_control` marker** — it just becomes a plain string. For Anthropic specifically, that means
zero caching, silently.

In [13]:
# --- Reference only: this is what a CACHEABLE Claude system prompt actually looks like ---
#
# from langchain_anthropic import ChatAnthropic
# from langchain_core.messages import SystemMessage
#
# BIG_STABLE_SYSTEM_PROMPT = "... a long, unchanging system prompt, e.g. 1800+ tokens ..."
#
# cacheable_system_message = SystemMessage(
#     content=[
#         {
#             "type": "text",
#             "text": BIG_STABLE_SYSTEM_PROMPT,
#             "cache_control": {"type": "ephemeral"},   # <-- this is the part the tuple shorthand can't express
#         }
#     ]
# )
#
# claude_model = ChatAnthropic(model="claude-sonnet-5")
# response = claude_model.invoke([cacheable_system_message, HumanMessage(content="{query}")])
#
# Check whether it worked via response.usage_metadata / response.response_metadata —
# look for cache_read / cache_creation token counts greater than zero.

print("Reference cell — not executed (no Anthropic key). Read the commented code above.")


Reference cell — not executed (no Anthropic key). Read the commented code above.


**Takeaway:** the tuple shorthand (`("system", "...")`) is the right default for learning and for
most small apps. The moment you're sending the *same* large system prompt or few-shot block on every
single call to Claude in a real product, switch that one message to the explicit `SystemMessage` +
`cache_control` form — it's a one-message change, not a rewrite, and it's often the single biggest
cost lever available to you.

---
## Real-world capstone: a support chatbot with consistent tone AND bounded memory

Putting it together: **few-shot examples** keep the tone consistent, **`MessagesPlaceholder` with
`n_messages`** stops the conversation from growing unbounded (and getting more expensive with every
turn), and the whole thing is one reusable template instead of hand-built message lists.

In [14]:
SUPPORT_EXAMPLES = [
    {"input": "My package is late",
     "output": "I'm sorry about the delay! Could you share your order number so I can look into it right away?"},
    {"input": "This product is broken",
     "output": "That's frustrating, I'm sorry. Let's get this sorted — can you tell me what's wrong with it?"},
]

example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=SUPPORT_EXAMPLES)

support_template = ChatPromptTemplate.from_messages([
    ("system", "You are a customer support agent. Match the warm, apologetic tone of the examples. "
               "Keep replies under 3 sentences and end with one clarifying question when useful."),
    few_shot,
    MessagesPlaceholder(variable_name="chat_history", optional=True, n_messages=6),  # last 6 turns max
    ("human", "{query}"),
])

support_chain = support_template | model  # reusing the ChatGroq model from Section 2


def run_support_chat(query: str, chat_history: list) -> str:
    reply = support_chain.invoke({"query": query, "chat_history": chat_history}).content
    chat_history.append(("human", query))
    chat_history.append(("ai", reply))
    return reply


history = []
print("Agent:", run_support_chat("Hey, my internet keeps dropping", history))
print("Agent:", run_support_chat("It's a Netgear router, about 2 years old", history))
print("\n--- chat_history now has", len(history), "messages, capped at 6 by n_messages ---")


Agent: I'm sorry for the disruption! Could you tell me if the lights on your modem are blinking or staying solid?
Agent: That's helpful, but let's first try a quick power cycle to see if it clears the glitch. Are you able to unplug the router for 30 seconds right now?

--- chat_history now has 4 messages, capped at 6 by n_messages ---


### Try it yourself
1. Run `run_support_chat` eight or nine times in a loop and print `len(history)` after each call —
   confirm it never exceeds 6 messages no matter how long the conversation runs.
2. Add a third few-shot example with a *different* tone on purpose (curt, no apology) and watch the
   agent's replies get inconsistent — this is exactly the failure mode few-shot examples are meant
   to prevent, so seeing it break is as instructive as seeing it work.
3. Swap `model` for the OpenAI model from `01_Models/01_chat_models.ipynb` and compare — does the
   tone-matching hold up as well?
4. If you ever get an Anthropic key: take the `SUPPORT_EXAMPLES` system prompt, make it deliberately
   long (500+ words), convert it to the `SystemMessage` + `cache_control` form from Section 7, and
   check `response.usage_metadata` on the second identical call for nonzero `cache_read` tokens.

---
**Next:** `03_Runnables/` — you've been piping `template | model` all notebook without asking what
that `|` actually does. Time to build a mini version of it from scratch.
